MCS 320 Project Two due by 2pm on Wednesday 22 July 2026.

The goal of the project is to generate code to apply algorithmic differentiation to polynomials in several variables.

# 1. Symbolic Gradient of a Product

We start with computing the gradient of the product of $n$ variables: 
$\displaystyle \prod_{i=1}^n x_i$.

In [1]:
n = 8

In [2]:
X = [var('x%d' % k) for k in range(1, n+1)]
X

[x1, x2, x3, x4, x5, x6, x7, x8]

In [3]:
pX = prod(X)
show(pX)

x1*x2*x3*x4*x5*x6*x7*x8

In [4]:
gX = [pX.diff(x) for x in X]
show(gX)

[x2*x3*x4*x5*x6*x7*x8,
 x1*x3*x4*x5*x6*x7*x8,
 x1*x2*x4*x5*x6*x7*x8,
 x1*x2*x3*x5*x6*x7*x8,
 x1*x2*x3*x4*x6*x7*x8,
 x1*x2*x3*x4*x5*x7*x8,
 x1*x2*x3*x4*x5*x6*x8,
 x1*x2*x3*x4*x5*x6*x7]

Now that we have the symbolic expression of the gradient of the product, we construct a tuple of consecutive positive integers for evaluation.

In [5]:
nt = tuple(k+1 for k in range(n))
nt

(1, 2, 3, 4, 5, 6, 7, 8)

To evaluate an expression in an arbitrary number $n$ of variables, we construct a dictionary with as keys the symbols for the variables and as values the corresponding numbers.  This dictionary is then used in the ``subs`` method applied to the expression.

In [6]:
xt = [{x : t} for (x, t) in zip(X, nt)]
xt

[{x1: 1}, {x2: 2}, {x3: 3}, {x4: 4}, {x5: 5}, {x6: 6}, {x7: 7}, {x8: 8}]

In [7]:
pX.subs(xt)

40320

In [8]:
[p.subs(xt) for p in gX]

[40320, 20160, 13440, 10080, 8064, 6720, 5760, 5040]

In [9]:
[pX.subs(xt)] + [p.subs(xt) for p in gX]

[40320, 40320, 20160, 13440, 10080, 8064, 6720, 5760, 5040]

In [10]:
timeit('[pX.subs(xt)] + [p.subs(xt) for p in gX]')

625 loops, best of 3: 160 μs per loop

Perhaps a Python function may run faster than the symbolic substitution.

In [11]:
def gradient_product(x):
    """
    Given in x is a list of values.
    Returns a list with as first component the product,
    and then the components of the gradient.
    """
    result = [product(v for v in x)]
    for k in range(len(x)):
        dk = product(v for v in x[:k])*product(v for v in x[k+1:])
        result.append(dk)
    return result

In [12]:
gradient_product(nt)

[40320, 40320, 20160, 13440, 10080, 8064, 6720, 5760, 5040]

In [13]:
timeit('gradient_product(nt)')

625 loops, best of 3: 18 μs per loop

The Python function runs indeed faster, and in SageMath, we can also evaluate it symbolically.

In [14]:
gradient_product(X)

[x1*x2*x3*x4*x5*x6*x7*x8,
 x2*x3*x4*x5*x6*x7*x8,
 x1*x3*x4*x5*x6*x7*x8,
 x1*x2*x4*x5*x6*x7*x8,
 x1*x2*x3*x5*x6*x7*x8,
 x1*x2*x3*x4*x6*x7*x8,
 x1*x2*x3*x4*x5*x7*x8,
 x1*x2*x3*x4*x5*x6*x8,
 x1*x2*x3*x4*x5*x6*x7]

Observe the repetitveness of the arithmetical operations: the sequence of the same $k$ products occurs multiple times.  Using the straightforward symbolic computation of the gradient of a product requires a number of multiplications that grows proportional to $n^2$, quadratically in the number of variables.

# 2. Reverse Mode of Algorithmic Differentiation

In the reverse mode of algorithmic differentiation, we use extra variables to accumuluate the forward products:

$$
   f_2 = x_1 \star x_2, f_i = f_{i-1} \star x_i, \mbox{ for } i=3,\ldots, n,
$$

so $f_n$ then holds the value of the product of the $n$ variables
and $f_{n-1}$ is the last component of the gradient.

The backward products are computed as

$$
   b_{n-1} = x_n \star x_{n-1}, 
   b_i = b_{i+1} \star x_i \mbox{ for } i=n-2, \ldots, 2,
$$

and we observe that $b_2$ equals the first component of the gradient.

The other components of the gradient are computed via the cross products:

$$
   c_1 = x_1 \star b_3, 
   c_i = f_i \star b_{i+2}, \mbox{ for } i=2, \ldots, n-3,
   c_{n-2} = f_{n-2} \star x_n.
$$

The number of multiplications equals $3 n - 5$.  The cost to evaluate and differentiate a product of $n$ variables is linear in $n$.

For $n=8$, the formulas are coded in a Python function.

In [15]:
def gradprod(x1, x2, x3, x4, x5, x6, x7, x8):
    """
    Returns the function values and the gradient
    of the product of the values in x.
    """
    f2 = x1*x2
    f3 = f2*x3
    f4 = f3*x4
    f5 = f4*x5
    f6 = f5*x6
    f7 = f6*x7
    f8 = f7*x8
    b7 = x8*x7
    b6 = b7*x6
    b5 = b6*x5
    b4 = b5*x4
    b3 = b4*x3
    b2 = b3*x2
    c1 = x1*b3
    c2 = f2*b4
    c3 = f3*b5
    c4 = f4*b6
    c5 = f5*b7
    c6 = f6*x8
    return (f8, b2, c1, c2, c3, c4, c5, c6, f7)

In [16]:
gradprod(*nt)

(40320, 40320, 20160, 13440, 10080, 8064, 6720, 5760, 5040)

In [17]:
gradient_product(nt)

[40320, 40320, 20160, 13440, 10080, 8064, 6720, 5760, 5040]

In [18]:
timeit('gradprod(*nt)')

625 loops, best of 3: 305 ns per loop

From the output of ``timeit()`` we observe a drop in magnitude, from microseconds to nanoseconds.

Evaluating the Python function in the sequence of symbols allows to verify the correctness.

In [19]:
gradprod(*X)

(x1*x2*x3*x4*x5*x6*x7*x8,
 x2*x3*x4*x5*x6*x7*x8,
 x1*x3*x4*x5*x6*x7*x8,
 x1*x2*x4*x5*x6*x7*x8,
 x1*x2*x3*x5*x6*x7*x8,
 x1*x2*x3*x4*x6*x7*x8,
 x1*x2*x3*x4*x5*x7*x8,
 x1*x2*x3*x4*x5*x6*x8,
 x1*x2*x3*x4*x5*x6*x7)

# 3. Code Generation

To apply the reverse mode to any number of variables, consider the definition of functions in strings, after stripping the documentation string.

In [20]:
strfun ="""
def strfungradprod(x1, x2, x3, x4, x5, x6, x7, x8):
    f2 = x1*x2
    f3 = f2*x3
    f4 = f3*x4
    f5 = f4*x5
    f6 = f5*x6
    f7 = f6*x7
    f8 = f7*x8
    b7 = x8*x7
    b6 = b7*x6
    b5 = b6*x5
    b4 = b5*x4
    b3 = b4*x3
    b2 = b3*x2
    c1 = x1*b3
    c2 = f2*b4
    c3 = f3*b5
    c4 = f4*b6
    c5 = f5*b7
    c6 = f6*x8
    return (f8, b2, c1, c2, c3, c4, c5, c6, f7)
"""

Executing the string via ``exec()`` defines the function.

In [21]:
exec(strfun)

In [22]:
strfungradprod(*nt)

(40320, 40320, 20160, 13440, 10080, 8064, 6720, 5760, 5040)

In [23]:
strfungradprod(*X)

(x1*x2*x3*x4*x5*x6*x7*x8,
 x2*x3*x4*x5*x6*x7*x8,
 x1*x3*x4*x5*x6*x7*x8,
 x1*x2*x4*x5*x6*x7*x8,
 x1*x2*x3*x5*x6*x7*x8,
 x1*x2*x3*x4*x6*x7*x8,
 x1*x2*x3*x4*x5*x7*x8,
 x1*x2*x3*x4*x5*x6*x8,
 x1*x2*x3*x4*x5*x6*x7)

## Assignment One

Take the formulas for the forward, backward, and cross products and write a function ``G(n)`` to generate a string for any number of variables, generalizing the code for $n=8$.  The input to ``G`` is ``n``, the number of variables.  The function ``G`` returns a string defining a Python function ``GP`` to evaluate a product of $n$ variables.  For $n=8$, the output ``GP`` should be the same as the above function.  In particular, work only with local variables to store the accumulated products.  Do not use lists or any other composite data structures.

Time the generated code for $n = 8, 16, 32, 64, 128, 256$ and format the times in a table.  Do you observe the $3 n$ progression of the times as $n$ increases?

## Assignment Two

Consider monomials in $n$ variables which are products of a selection of $k$ variables:

$$
   \prod_{j=1}^k x_{i_j}
$$

for a set of indices $\{i_1, i_2, \ldots, i_k \} \subset \{1,2,\ldots, n\}$.

Extend the code to generate ``GP`` with an extra argument: the list of indices $i_1, i_2, \ldots, i_k$.

Time the execution for $k=8, 16, 32, 64, 128, 256$, where $n = 2k$, for random selections of $k$ indices.  Format the times in a table.  Do you observe the $3k$ progression of the times as $k$ increases?

## Assignment Three

Consider a polynomial of $m$ monomials in $n$ variables, where every monomial is a product of variables:

$$
   p = \sum_{i=1}^m c_i \prod_{j=1}^{k_i} x_{i_j},
$$

where the coefficients $c_i$ are nonzero integer numbers.

Extend the code to generate ``GP`` to evaluate and differentiate $p$.  The coefficients and monomials are given in two lists, respectively a list of integers and a list of tuples with the indices.

Generate random polynomials where $m = n$, and where the number of variables in each monomial is $n/2$, for $n=8, 16, 32, 64, 128, 256$.  Compare the evaluate a random integers with the symbolic substitution.  Format the times of the symbolic substitution and the algorithmic differentiation in a table.  Describe your findings.

## 4. The Deadline is Wednesday 22 July, at 2pm.

Upload your answer to gradescope at the latest on Wednesday 22 July,
before 2pm.

The solution consists of one single notebook,
organized according to the assignments.
Mark the start of each solution to an assignment using a heading in a cell.
Your notebook should run from top to bottom as a program without errors.
Apply proper formatting in your notebook so it reads
like a technical report if you would print it.
Document the execution cells with complete sentences,
properly formatted in markdown cells.

You may (not must) work in pairs for this project.  
A pair consists of two, not three or more.
If you decide to work in a pair,
then you must send me an email with the name of your partner and
with the email address of your partner in the copy of the email,
before 2pm on Friday 17 July.
If working in a pair, then only one Jupyter notebook should be submitted.

If you have questions, concerns, or difficulties,
feel free to contact me for help.